# C-MAPSS FD001–FD004 — Tune one combined XGBoost model

**Goal:** Optimize one XGBoost model across all four datasets, compare it fairly with previous combined and four-specialist approaches, and log results **both locally and to DagsHub MLflow**.
**Evaluation rules**
- Split by engine within each FD dataset, using the same `random_state=42` and 20% validation split as the previous notebook.
- Tune hyperparameters with **3-fold engine-group cross-validation on the training portion only**.
- Compare the tuned model against the earlier models on the **same untouched validation portion**.
- Local logs and DagsHub MLflow runs are separate; the notebook verifies the remote tracking URI explicitly.

## 1. Imports, paths, and local logging

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from itertools import product
import csv
import json
import logging
import os
import joblib
import numpy as np
import pandas as pd
import mlflow
import dagshub
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, ParameterSampler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

# Locate the project root from either notebooks/ or the repository root.
ROOT = Path("..").resolve() if Path("../data/curated/train").exists() else Path(".").resolve()
DATA_DIR = ROOT / "data" / "curated" / "train"
DATASETS = ["FD001", "FD002", "FD003", "FD004"]
SEED = 42
N_TRIALS = 12
CV_FOLDS = 3

# Every execution gets its own folder; previous logs and models remain untouched.
RUN_NAME = datetime.now(timezone.utc).strftime("combined_xgb_%Y%m%d_%H%M%S")
RUN_DIR = ROOT / "reports" / "combined_xgboost_tuning" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
TRIALS_CSV = RUN_DIR / "cv_trials.csv"

# Log progress to a local file AND to the notebook output.
logger = logging.getLogger("combined_xgb_tuning")
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(RUN_DIR / "tuning.log", encoding="utf-8")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)
stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)
logger.propagate = False

logger.info("Local experiment folder: %s", RUN_DIR)
logger.info("Curated training folder exists: %s", DATA_DIR.exists())
assert DATA_DIR.is_dir(), "Cannot find data/curated/train. Pull your DVC data first."

c:\Users\omeshamisu.anigala\git\predictive-maintenance-mlops\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-27 13:23:35,194 | INFO | Local experiment folder: C:\Users\omeshamisu.anigala\git\predictive-maintenance-mlops\reports\combined_xgboost_tuning\combined_xgb_20261027_182335
2026-10-27 13:23:35,194 | INFO | Curated training folder exists: True


## 2. Load all four datasets and recreate the original engine-level validation split

In [2]:
# Load only TRAIN data; NASA test data is intentionally not loaded here.
raw_train = {fd: pd.read_parquet(DATA_DIR / f"train_{fd}.parquet") for fd in DATASETS}

# Match the exact feature schema used by our previous individual-vs-combined notebook.
base_features = [c for c in raw_train["FD001"].columns if c not in ["unit_number", "RUL"]]
indicator_features = [f"dataset_{fd}" for fd in DATASETS]
feature_columns = base_features + indicator_features

train_parts = []
val_parts = []
for fd in DATASETS:
    df = raw_train[fd].copy()
    assert {"unit_number", "time_in_cycles", "RUL"}.issubset(df.columns), f"Missing required columns in {fd}"
    assert set(base_features).issubset(df.columns), f"Feature mismatch in {fd}"
    for name in DATASETS:
        df[f"dataset_{name}"] = int(fd == name)
    df["dataset_id"] = fd
    df["engine_key"] = fd + ":" + df["unit_number"].astype(str)

    # Same split as the previous notebook: 80% of engines train, 20% validate.
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    train_idx, val_idx = next(splitter.split(df, groups=df["unit_number"]))
    train_df = df.iloc[train_idx].copy()
    val_df = df.iloc[val_idx].copy()
    assert set(train_df["engine_key"]).isdisjoint(set(val_df["engine_key"]))
    train_parts.append(train_df)
    val_parts.append(val_df)
    logger.info("%s: %s training engines, %s validation engines", fd, train_df["unit_number"].nunique(), val_df["unit_number"].nunique())

train_df = pd.concat(train_parts, ignore_index=True)
val_df = pd.concat(val_parts, ignore_index=True)
X_train = train_df[feature_columns]
y_train = train_df["RUL"]
X_val = val_df[feature_columns]
y_val = val_df["RUL"]

assert X_train.columns.tolist() == X_val.columns.tolist()
assert X_train.notna().all().all() and X_val.notna().all().all(), "Missing feature values detected."
logger.info("Training rows: %s | Validation rows: %s | Features: %s", len(X_train), len(X_val), len(feature_columns))

2026-10-27 13:23:41,359 | INFO | FD001: 80 training engines, 20 validation engines
2026-10-27 13:23:41,447 | INFO | FD002: 208 training engines, 52 validation engines
2026-10-27 13:23:41,488 | INFO | FD003: 80 training engines, 20 validation engines
2026-10-27 13:23:41,583 | INFO | FD004: 199 training engines, 50 validation engines
2026-10-27 13:23:41,607 | INFO | Training rows: 129331 | Validation rows: 31028 | Features: 29


## 3. Connect to **DagsHub**, not a local SQLite MLflow database

This is the crucial fix from the previous notebook. We set and **verify** the remote tracking URL *before* creating any MLflow runs.

If authentication is needed, DagsHub may ask you to sign in. Local CSV/log files will still be saved if remote logging fails.

In [3]:
# Initialize DagsHub authentication and point MLflow at the REMOTE server.
dagshub.init(repo_owner="Omesham", repo_name="predictive-maintenance-mlops", mlflow=True)
REMOTE_URI = "https://dagshub.com/Omesham/predictive-maintenance-mlops.mlflow/"
mlflow.set_tracking_uri(REMOTE_URI)

# Never accidentally write this experiment into notebooks/mlflow.db.
assert mlflow.get_tracking_uri().rstrip("/") == REMOTE_URI.rstrip("/"), "MLflow is NOT pointing at DagsHub."
mlflow.set_experiment("FD001_FD004_Combined_XGBoost_Tuning")
logger.info("MLflow tracking URI: %s", mlflow.get_tracking_uri())

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

c:\Users\omeshamisu.anigala\git\predictive-maintenance-mlops\venv\Lib\site-packages\rich\live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=ed5d9fb1-b24d-4d00-857e-b8d9a529bb4a&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=74874d0d030a54eb7e893a4672fa54fade9cbd05eeec1b4f28d19b1642675c26




Accessing as Omesham

Initialized MLflow to track repo "Omesham/predictive-maintenance-mlops"

Repository Omesham/predictive-maintenance-mlops initialized!

2026/10/27 13:24:58 INFO mlflow.tracking.fluent: Experiment with name 'FD001_FD004_Combined_XGBoost_Tuning' does not exist. Creating a new experiment.
2026-10-27 13:25:06,024 | INFO | MLflow tracking URI: https://dagshub.com/Omesham/predictive-maintenance-mlops.mlflow/


## 4. Hyperparameter search using 3-fold grouped cross-validation

**What we're tuning:** number of trees, tree depth, learning rate, subsampling, and regularization.

**Numeric example:** If an engine has 180 cycles, all 180 rows for that engine stay together in one fold. The model never trains on cycle 1 of an engine and validates on cycle 180 of that *same* engine.

**Runtime:** 12 configurations × 3 folds = **36 XGBoost fits**, plus one validation fit and the final deployment refit. This can take several minutes. Each finished trial is written to a local CSV immediately and logged to MLflow. To start smaller, change `N_TRIALS` to 6 before running.

In [4]:
# Keep parameter choices explicit and reproducible.
search_space = {
    "n_estimators": [200, 350, 500],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.03, 0.05, 0.1],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "reg_lambda": [1.0, 5.0, 10.0],
}
candidates = list(ParameterSampler(search_space, n_iter=N_TRIALS, random_state=SEED))

# Engine IDs overlap across FD datasets; dataset-prefixed keys make them unique.
cv = GroupKFold(n_splits=CV_FOLDS)
cv_indices = list(cv.split(X_train, y_train, groups=train_df["engine_key"]))

# Confirm that no engine appears on both sides of a CV fold.
for fit_idx, score_idx in cv_indices:
    assert set(train_df.iloc[fit_idx]["engine_key"]).isdisjoint(set(train_df.iloc[score_idx]["engine_key"]))

def build_xgb(params):
    # A single new model per fold; n_jobs uses local CPU cores.
    return XGBRegressor(**params, objective="reg:squarederror", random_state=SEED, n_jobs=-1, tree_method="hist")

def score_predictions(actual, predicted):
    return {
        "mae": float(mean_absolute_error(actual, predicted)),
        "rmse": float(np.sqrt(mean_squared_error(actual, predicted))),
    }

logger.info("Prepared %s configurations and %s engine-safe folds.", len(candidates), len(cv_indices))

2026-10-27 13:30:51,921 | INFO | Prepared 12 configurations and 3 engine-safe folds.


## 5. Run tuning — save every result locally and log each trial remotely

In [5]:
# This is a NEW tuning experiment. Do not rerun this cell unless you intend a new search.
trial_records = []
for trial_no, params in enumerate(candidates, start=1):
    fold_mae = []
    fold_rmse = []
    logger.info("Starting trial %s/%s: %s", trial_no, len(candidates), params)

    for fold_no, (fit_idx, score_idx) in enumerate(cv_indices, start=1):
        estimator = build_xgb(params)
        estimator.fit(X_train.iloc[fit_idx], y_train.iloc[fit_idx])
        predictions = estimator.predict(X_train.iloc[score_idx])
        scores = score_predictions(y_train.iloc[score_idx], predictions)
        fold_mae.append(scores["mae"])
        fold_rmse.append(scores["rmse"])
        logger.info("Trial %s fold %s: MAE %.3f, RMSE %.3f", trial_no, fold_no, scores["mae"], scores["rmse"])
        del estimator

    record = {
        "trial": trial_no,
        **params,
        "cv_mae": float(np.mean(fold_mae)),
        "cv_rmse": float(np.mean(fold_rmse)),
        "cv_mae_std": float(np.std(fold_mae)),
    }
    trial_records.append(record)

    # Write the completed trial to a local CSV immediately, even if DagsHub is unavailable.
    with TRIALS_CSV.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(record.keys()))
        if handle.tell() == 0:
            writer.writeheader()
        writer.writerow(record)

    # Log metrics and parameters, NOT large model artifacts, for each trial.
    try:
        with mlflow.start_run(run_name=f"{RUN_NAME}_trial_{trial_no:02d}"):
            mlflow.set_tags({"stage": "cross_validation", "approach": "combined", "dataset": "FD001-FD004", "local_run_name": RUN_NAME})
            mlflow.log_params({**params, "random_state": SEED, "cv_folds": CV_FOLDS})
            mlflow.log_metrics({"cv_mae": record["cv_mae"], "cv_rmse": record["cv_rmse"], "cv_mae_std": record["cv_mae_std"]})
    except Exception as exc:
        logger.warning("Remote MLflow logging failed for trial %s; local CSV is safe: %s", trial_no, exc)

    logger.info("Finished trial %s/%s: mean CV MAE %.3f", trial_no, len(candidates), record["cv_mae"])

trial_table = pd.DataFrame(trial_records).sort_values(["cv_mae", "cv_rmse"]).reset_index(drop=True)
trial_table.to_csv(RUN_DIR / "cv_trials_sorted.csv", index=False)
display(trial_table.round(3))

2026-10-27 13:31:04,553 | INFO | Starting trial 1/12: {'subsample': 1.0, 'reg_lambda': 10.0, 'n_estimators': 350, 'min_child_weight': 1, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 1.0}
2026-10-27 13:31:05,881 | INFO | Trial 1 fold 1: MAE 32.999, RMSE 48.305
2026-10-27 13:31:07,480 | INFO | Trial 1 fold 2: MAE 32.166, RMSE 46.528
2026-10-27 13:31:08,780 | INFO | Trial 1 fold 3: MAE 31.677, RMSE 46.453


🏃 View run combined_xgb_20261027_182335_trial_01 at: https://dagshub.com/Omesham/predictive-maintenance-mlops.mlflow/#/experiments/2/runs/0b8e743460d742a3b02846f6e24cfd72
🧪 View experiment at: https://dagshub.com/Omesham/predictive-maintenance-mlops.mlflow/#/experiments/2


2026-10-27 13:31:21,909 | INFO | Finished trial 1/12: mean CV MAE 32.281
2026-10-27 13:31:21,909 | INFO | Starting trial 2/12: {'subsample': 0.8, 'reg_lambda': 10.0, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 5, 'learning_rate': 0.03, 'colsample_bytree': 0.8}
2026-10-27 13:31:23,233 | INFO | Trial 2 fold 1: MAE 33.382, RMSE 48.201
2026-10-27 13:31:24,497 | INFO | Trial 2 fold 2: MAE 32.050, RMSE 46.033
2026-10-27 13:31:25,854 | INFO | Trial 2 fold 3: MAE 31.942, RMSE 46.002
2026-10-27 13:31:30,987 | INFO | Finished trial 2/12: mean CV MAE 32.458
2026-10-27 13:31:30,987 | INFO | Starting trial 3/12: {'subsample': 1.0, 'reg_lambda': 5.0, 'n_estimators': 350, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 1.0}


🏃 View run combined_xgb_20261027_182335_trial_02 at: https://dagshub.com/Omesham/predictive-maintenance-mlops.mlflow/#/experiments/2/runs/7059265ee5c84dc685586669edc6aaa1
🧪 View experiment at: https://dagshub.com/Omesham/predictive-maintenance-mlops.mlflow/#/experiments/2


2026-10-27 13:31:31,711 | INFO | Trial 3 fold 1: MAE 34.081, RMSE 48.942
2026-10-27 13:31:32,557 | INFO | Trial 3 fold 2: MAE 32.449, RMSE 46.322
2026-10-27 13:31:33,493 | INFO | Trial 3 fold 3: MAE 32.459, RMSE 46.214
2026-10-27 13:31:36,443 | INFO | Finished trial 3/12: mean CV MAE 32.996
2026-10-27 13:31:36,443 | INFO | Starting trial 4/12: {'subsample': 1.0, 'reg_lambda': 10.0, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 7, 'learning_rate': 0.1, 'colsample_bytree': 0.8}


🏃 View run combined_xgb_20261027_182335_trial_03 at: https://dagshub.com/Omesham/predictive-maintenance-mlops.mlflow/#/experiments/2/runs/a813c02c13ea46bb954008ec0355a013
🧪 View experiment at: https://dagshub.com/Omesham/predictive-maintenance-mlops.mlflow/#/experiments/2


... (16 outputs left out)

2026-10-27 13:32:38,022 | INFO | Finished trial 11/12: mean CV MAE 32.554
2026-10-27 13:32:38,022 | INFO | Starting trial 12/12: {'subsample': 1.0, 'reg_lambda': 5.0, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
2026-10-27 13:32:38,517 | INFO | Trial 12 fold 1: MAE 33.363, RMSE 48.352
2026-10-27 13:32:38,969 | INFO | Trial 12 fold 2: MAE 32.246, RMSE 46.431
2026-10-27 13:32:39,465 | INFO | Trial 12 fold 3: MAE 32.001, RMSE 46.361


🏃 View run combined_xgb_20261027_182335_trial_12 at: https://dagshub.com/Omesham/predictive-maintenance-mlops.mlflow/#/experiments/2/runs/5f141688a372491c8d56b87f39f2e573
🧪 View experiment at: https://dagshub.com/Omesham/predictive-maintenance-mlops.mlflow/#/experiments/2


2026-10-27 13:32:50,117 | INFO | Finished trial 12/12: mean CV MAE 32.537


,trial,subsample,reg_lambda,n_estimators,min_child_weight,max_depth,learning_rate,colsample_bytree,cv_mae,cv_rmse,cv_mae_std
0,7,1.0,1.0,350,1,7,0.05,0.8,32.147,46.954,0.640
1,8,0.8,10.0,350,1,7,0.05,0.8,32.150,46.932,0.638
2,5,0.8,5.0,500,5,7,0.03,1.0,32.210,46.963,0.577
3,1,1.0,10.0,350,1,7,0.05,1.0,32.281,47.095,0.546
4,4,1.0,10.0,500,5,7,0.10,0.8,32.397,47.507,0.463
5,10,1.0,1.0,350,3,5,0.10,1.0,32.426,47.086,0.577
6,2,0.8,10.0,500,1,5,0.03,0.8,32.458,46.745,0.655
7,12,1.0,5.0,200,1,5,0.10,1.0,32.537,47.048,0.593
8,11,0.8,5.0,500,1,5,0.03,1.0,32.554,46.904,0.589
9,9,1.0,1.0,500,5,7,0.10,1.0,32.619,47.829,0.449


## 6. Evaluate the best CV configuration on our **original held-out validation engines**

This is the first time these engines are used for the tuned model. The earlier notebook reported the following **rounded** validation results on the same split:

- Previous combined approach: **MAE 32.918, RMSE 46.383**
- Four individual models: **MAE 32.941, RMSE 46.903**

We use these as comparison references, not as training targets. Do **not** use NASA test results to choose the winner.

In [6]:
# Choose hyperparameters using cross-validation scores ONLY.
best_trial = trial_table.iloc[0].to_dict()
best_params = {key: best_trial[key] for key in search_space}
best_params["n_estimators"] = int(best_params["n_estimators"])
best_params["max_depth"] = int(best_params["max_depth"])
best_params["min_child_weight"] = int(best_params["min_child_weight"])

# Train on the original 80%-engine training split; score on the held-out 20%.
tuned_validation_model = build_xgb(best_params)
tuned_validation_model.fit(X_train, y_train)
val_predictions = tuned_validation_model.predict(X_val)
tuned_val = score_predictions(y_val, val_predictions)

comparison = pd.DataFrame([
    {"approach": "Previous combined model", "val_mae": 32.918, "val_rmse": 46.383},
    {"approach": "Four individual models", "val_mae": 32.941, "val_rmse": 46.903},
    {"approach": "Tuned combined XGBoost", "val_mae": tuned_val["mae"], "val_rmse": tuned_val["rmse"]},
]).sort_values(["val_mae", "val_rmse"]).reset_index(drop=True)

comparison.to_csv(RUN_DIR / "validation_comparison.csv", index=False)
joblib.dump(tuned_validation_model, RUN_DIR / "tuned_validation_model.joblib")
logger.info("Tuned validation MAE: %.3f | RMSE: %.3f", tuned_val["mae"], tuned_val["rmse"])
display(comparison.round(3))

2026-10-27 13:35:28,268 | INFO | Tuned validation MAE: 32.391 | RMSE: 46.343


,approach,val_mae,val_rmse
0,Tuned combined XGBoost,32.391,46.343
1,Previous combined model,32.918,46.383
2,Four individual models,32.941,46.903


## 7. Select the deployment approach and save a deployable model

**Selection rule:** Lowest pooled validation MAE. If tuning does not beat the previous combined model, keep the previous model; do not call the new model the winner.

This cell saves a **new combined XGBoost candidate** refitted on all four original training datasets, plus metadata for the CSV-upload API. It does **not** overwrite your previous `models/champion/` files or register a model.

If the tuned model is not the validation winner, the output says so clearly.

In [7]:
# Determine whether tuning actually improved the previous combined baseline.
tuned_is_best = tuned_val["mae"] < min(32.918, 32.941)
selected_approach = "tuned_combined_xgboost" if tuned_is_best else "previous_combined_model"
logger.info("Selected using validation MAE: %s", selected_approach)

# Refit the tuned candidate on ALL original training engines for possible deployment.
# Do not use the NASA test dataset for this refit.
full_train = pd.concat([raw_train[fd].assign(
    **{f"dataset_{name}": int(fd == name) for name in DATASETS}
) for fd in DATASETS], ignore_index=True)
X_full = full_train[feature_columns]
y_full = full_train["RUL"]
deployment_candidate = build_xgb(best_params)
deployment_candidate.fit(X_full, y_full)

# Save a local model and exact input schema, without replacing prior champions.
candidate_path = RUN_DIR / "combined_xgboost_tuned_full_train.joblib"
joblib.dump(deployment_candidate, candidate_path)
metadata = {
    "candidate_type": "combined_xgboost_tuned",
    "selected_using_validation": selected_approach,
    "model_file": candidate_path.name,
    "dataset_ids": DATASETS,
    "input_features": base_features,
    "model_features": feature_columns,
    "indicator_features": indicator_features,
    "best_params": best_params,
    "validation_metrics": tuned_val,
    "cv_best_mae": float(best_trial["cv_mae"]),
    "training_rows": len(full_train),
    "test_set_used": False,
    "warning": "Joblib files must be loaded only from trusted sources.",
}
(RUN_DIR / "model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
logger.info("Saved candidate: %s", candidate_path)

2026-10-27 13:35:52,988 | INFO | Selected using validation MAE: tuned_combined_xgboost
2026-10-27 13:35:55,028 | INFO | Saved candidate: C:\Users\omeshamisu.anigala\git\predictive-maintenance-mlops\reports\combined_xgboost_tuning\combined_xgb_20261027_182335\combined_xgboost_tuned_full_train.joblib


## 8. Log final comparison and ONE model artifact to DagsHub MLflow

We upload only the **new tuned candidate**, not all 12 trial models. This avoids the long upload delays you experienced before.

MLflow may take a few minutes to upload this single artifact. The model and metrics are **already saved locally**, so a network failure will not erase them.

**No model registration yet.** We will register the actual winner after reviewing the comparison.

In [8]:
# A separate summary run holds the comparison, exact parameters, and trained candidate.
try:
    with mlflow.start_run(run_name=f"{RUN_NAME}_final_comparison") as final_run:
        mlflow.set_tags({
            "stage": "validation_comparison",
            "approach": "combined",
            "selected_approach": selected_approach,
            "selection_metric": "validation_mae",
            "local_run_name": RUN_NAME,
        })
        mlflow.log_params({**best_params, "cv_folds": CV_FOLDS, "random_state": SEED})
        mlflow.log_metrics({
            "best_cv_mae": float(best_trial["cv_mae"]),
            "tuned_val_mae": tuned_val["mae"],
            "tuned_val_rmse": tuned_val["rmse"],
            "previous_combined_val_mae": 32.918,
            "previous_individual_val_mae": 32.941,
        })
        mlflow.log_artifact(str(TRIALS_CSV), artifact_path="reports")
        mlflow.log_artifact(str(RUN_DIR / "validation_comparison.csv"), artifact_path="reports")
        mlflow.log_artifact(str(RUN_DIR / "model_metadata.json"), artifact_path="model")
        mlflow.log_artifact(str(candidate_path), artifact_path="model")
        logger.info("DagsHub MLflow summary run: %s", final_run.info.run_id)
except Exception as exc:
    logger.warning("Remote summary upload failed; local model and reports remain saved: %s", exc)

print("Local results:", RUN_DIR)
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Validation winner:", comparison.iloc[0]["approach"])
print("Tuned combined MAE:", round(tuned_val["mae"], 3))
print("Tuned combined RMSE:", round(tuned_val["rmse"], 3))

2026-10-27 13:36:15,800 | INFO | DagsHub MLflow summary run: 8e03a02a0d05445abc98d0a6253fe5d4


🏃 View run combined_xgb_20261027_182335_final_comparison at: https://dagshub.com/Omesham/predictive-maintenance-mlops.mlflow/#/experiments/2/runs/8e03a02a0d05445abc98d0a6253fe5d4
🧪 View experiment at: https://dagshub.com/Omesham/predictive-maintenance-mlops.mlflow/#/experiments/2
Local results: C:\Users\omeshamisu.anigala\git\predictive-maintenance-mlops\reports\combined_xgboost_tuning\combined_xgb_20261027_182335
MLflow tracking URI: https://dagshub.com/Omesham/predictive-maintenance-mlops.mlflow/
Validation winner: Tuned combined XGBoost
Tuned combined MAE: 32.391
Tuned combined RMSE: 46.343


In [11]:
import pandas as pd

for fd in ["FD001", "FD002", "FD003", "FD004"]:
    df = pd.read_parquet(f"../data/curated/test/test_{fd}.parquet")
    print(fd, df.shape, df.columns.tolist())

FD001 (13096, 27) ['unit_number', 'time_in_cycles', 'operational_settings1', 'operational_settings2', 'operational_settings3', 'sensor_measurement1', 'sensor_measurement2', 'sensor_measurement3', 'sensor_measurement4', 'sensor_measurement5', 'sensor_measurement6', 'sensor_measurement7', 'sensor_measurement8', 'sensor_measurement9', 'sensor_measurement10', 'sensor_measurement11', 'sensor_measurement12', 'sensor_measurement13', 'sensor_measurement14', 'sensor_measurement15', 'sensor_measurement16', 'sensor_measurement17', 'sensor_measurement18', 'sensor_measurement19', 'sensor_measurement20', 'sensor_measurement21', 'RUL']
FD002 (33991, 27) ['unit_number', 'time_in_cycles', 'operational_settings1', 'operational_settings2', 'operational_settings3', 'sensor_measurement1', 'sensor_measurement2', 'sensor_measurement3', 'sensor_measurement4', 'sensor_measurement5', 'sensor_measurement6', 'sensor_measurement7', 'sensor_measurement8', 'sensor_measurement9', 'sensor_measurement10', 'sensor_measu

In [12]:
test_frames = []

for fd in ["FD001", "FD002", "FD003", "FD004"]:
    df = pd.read_parquet(
        f"../data/curated/test/test_{fd}.parquet"
    )

    # Keep the last recorded cycle for each engine
    df = (df.sort_values("time_in_cycles").groupby("unit_number").tail(1).copy())

    # Add dataset indicators
    for dataset in ["FD001", "FD002", "FD003", "FD004"]:
        df[f"dataset_{dataset}"] = int(fd == dataset)

    df["dataset"] = fd
    test_frames.append(df)

final_test = pd.concat(test_frames, ignore_index=True)

X_test = final_test[X_train.columns]
y_test = final_test["RUL"]

print("Engines:", len(final_test))
print("Features:", X_test.shape[1])

Engines: 707
Features: 29


In [13]:
import joblib
from pathlib import Path

model_path = next(Path("../reports/combined_xgboost_tuning").glob("*/combined_xgboost_tuned_full_train.joblib"))
model = joblib.load(model_path)

print("Model loaded successfully!")

Model loaded successfully!


In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Final Test MAE: {mae:.3f}")
print(f"Final Test RMSE: {rmse:.3f}")

Final Test MAE: 23.425
Final Test RMSE: 33.268


In [16]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("https://dagshub.com/Omesham/predictive-maintenance-mlops.mlflow/")

with mlflow.start_run(run_name="final_tuned_xgboost_registration"):
    mlflow.log_metric("test_mae", 23.425)
    mlflow.log_metric("test_rmse", 33.268)
    mlflow.xgboost.log_model(model, name="model", registered_model_name="CMapss_RUL_XGBoost")

Successfully registered model 'CMapss_RUL_XGBoost'.
2026/10/27 14:05:51 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: CMapss_RUL_XGBoost, version 1
Created version '1' of model 'CMapss_RUL_XGBoost'.


🏃 View run final_tuned_xgboost_registration at: https://dagshub.com/Omesham/predictive-maintenance-mlops.mlflow/#/experiments/2/runs/d77be9349b9f452183be0fd50d2f6075
🧪 View experiment at: https://dagshub.com/Omesham/predictive-maintenance-mlops.mlflow/#/experiments/2


In [17]:
from mlflow.models import infer_signature
import mlflow.xgboost

signature = infer_signature(X_train.head(5), model.predict(X_train.head(5)))

with mlflow.start_run(run_name="xgboost_with_signature"):
    mlflow.xgboost.log_model(
        model,
        name="model",
        signature=signature,
        registered_model_name="CMapss_RUL_XGBoost"
    )

c:\Users\omeshamisu.anigala\git\predictive-maintenance-mlops\venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'CMapss_RUL_XGBoost' already exists. Creating a new version of this model...
2026/10/27 14:11:42 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model versi

🏃 View run xgboost_with_signature at: https://dagshub.com/Omesham/predictive-maintenance-mlops.mlflow/#/experiments/2/runs/b95c632304344b6aaec3fdd939df5f6e
🧪 View experiment at: https://dagshub.com/Omesham/predictive-maintenance-mlops.mlflow/#/experiments/2


In [18]:
from mlflow import MlflowClient

client = MlflowClient()
version = client.get_model_version("CMapss_RUL_XGBoost", "3")
print(mlflow.models.get_model_info(version.source).signature)

inputs: 
  ['time_in_cycles': long (required), 'operational_settings1': double (required), 'operational_settings2': double (required), 'operational_settings3': double (required), 'sensor_measurement1': double (required), 'sensor_measurement2': double (required), 'sensor_measurement3': double (required), 'sensor_measurement4': double (required), 'sensor_measurement5': double (required), 'sensor_measurement6': double (required), 'sensor_measurement7': double (required), 'sensor_measurement8': double (required), 'sensor_measurement9': double (required), 'sensor_measurement10': double (required), 'sensor_measurement11': double (required), 'sensor_measurement12': double (required), 'sensor_measurement13': double (required), 'sensor_measurement14': double (required), 'sensor_measurement15': double (required), 'sensor_measurement16': double (required), 'sensor_measurement17': long (required), 'sensor_measurement18': long (required), 'sensor_measurement19': double (required), 'sensor_measureme

In [19]:
client.set_registered_model_alias("CMapss_RUL_XGBoost", "champion", "3")